# deep research using google search: practice

This module/project pack is extracted from the local files listed below. Functions and line numbers are source locations; the original program is not executed by practice.

Use this with [the walkthrough](walkthrough.md).

**Source:**

- [2_openai/community_contributions/deep_research_using_google_search/deep_research.py](../../../../../../2_openai/community_contributions/deep_research_using_google_search/deep_research.py)
- [2_openai/community_contributions/deep_research_using_google_search/email_agent.py](../../../../../../2_openai/community_contributions/deep_research_using_google_search/email_agent.py)
- [2_openai/community_contributions/deep_research_using_google_search/google_search_agent.py](../../../../../../2_openai/community_contributions/deep_research_using_google_search/google_search_agent.py)
- [2_openai/community_contributions/deep_research_using_google_search/llm_as_judge.py](../../../../../../2_openai/community_contributions/deep_research_using_google_search/llm_as_judge.py)
- [2_openai/community_contributions/deep_research_using_google_search/planner_agent.py](../../../../../../2_openai/community_contributions/deep_research_using_google_search/planner_agent.py)
- [2_openai/community_contributions/deep_research_using_google_search/README.md](../../../../../../2_openai/community_contributions/deep_research_using_google_search/README.md)
- [2_openai/community_contributions/deep_research_using_google_search/research_manager.py](../../../../../../2_openai/community_contributions/deep_research_using_google_search/research_manager.py)
- [2_openai/community_contributions/deep_research_using_google_search/search_agent.py](../../../../../../2_openai/community_contributions/deep_research_using_google_search/search_agent.py)
- [2_openai/community_contributions/deep_research_using_google_search/writer_agent.py](../../../../../../2_openai/community_contributions/deep_research_using_google_search/writer_agent.py)

**Run one cell at a time:** predict → run with Shift+Enter → explain → change one thing. Select the root project .venv kernel.

**Offline adaptation:** code below uses fictional data or local libraries. It never executes the linked source, reads a credential file, sends messages, starts a UI/server, or changes the course databases. Simulations are labeled; they do not verify external integrations.

## 1. Configuration stays outside executable examples

Environment-variable names identify settings. Values belong in .environment/.env and are loaded deliberately. This exercise inspects only the path and simulated presence flags; it reads no private file or variable values.

**Predict:** Does finding a file prove that authentication works?

In [ ]:
from pathlib import Path
# Find the repository from a kernel started inside it.
root = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "1_foundations" / "3_lab3.ipynb").is_file()), None)
if root is None:
    raise FileNotFoundError("Start your kernel inside ai_agents")
configuration_path = root / ".environment" / ".env"
print("Configuration path:", configuration_path)
simulated_presence = {"OPENAI_API_KEY": True, "GOOGLE_API_KEY": False}
print(simulated_presence)  # Booleans only; these are fictional flags.
assert configuration_path.name == ".env"

**Check:** No. File existence, variable presence, and successful provider authentication are separate checks.

**Change one thing:** Explain the difference between a missing file, a missing variable, and a provider rejecting authentication. Do not print token prefixes or values.

## 2. Configure a real SDK agent and inspect its tool schema

Agent holds instructions/model/tools. The function_tool decorator constructs a schema from the signature and docstring. Constructing this configuration does not run the agent or contact a provider.

**Predict:** Which argument names will appear in the registered tool schema?

In [ ]:
from agents import Agent, function_tool, set_tracing_disabled
set_tracing_disabled(True)
@function_tool
def add_numbers(a: int, b: int) -> int:
    """Add two integers."""
    return a + b
agent = Agent(name="Practice teacher", instructions="Use add_numbers for addition.",
              model="gpt-4o-mini", tools=[add_numbers])
print(agent.name, agent.tools[0].name)
print(sorted(agent.tools[0].params_json_schema["properties"]))
assert sorted(agent.tools[0].params_json_schema["properties"]) == ["a", "b"]

**Check:** a and b. The schema advertises the function; Runner executes the model/tool workflow when deliberately invoked.

**Change one thing:** Change the docstring and inspect the tool description. Find Runner.run in the source: that is where the interaction loop actually begins.

## 3. Client, request, and response are separate objects

Constructing a client prepares a connection interface. A request invokes it. The returned response contains choices, a message, and its content. This fixture imitates that nested shape.

**Predict:** What does choices[0] select?

In [ ]:
from types import SimpleNamespace
response = SimpleNamespace(choices=[
    SimpleNamespace(message=SimpleNamespace(content="The answer is 4."))
])
answer = response.choices[0].message.content
print(answer)
assert answer == "The answer is 4."

**Check:** The first returned choice. This is simulated response data, not a model call.

**Change one thing:** Add a second choice with different text. Read choices[1] and explain why choosing an index matters.

## 4. A tool schema describes; a dispatcher executes

The model can request a function call. Your application decodes arguments, recognizes the name, executes the function, and returns a matching tool result. No model is needed to practice this boundary.

**Predict:** Does defining the schema call the function?

In [ ]:
import json
records = []
def record_interest(email):
    records.append(email)  # In memory only; does not send email or notifications.
    return {"recorded": True}
schema = {
    "type": "function", "function": {
        "name": "record_interest", "description": "Record a supplied address",
        "parameters": {"type": "object", "properties": {"email": {"type": "string"}},
                       "required": ["email"], "additionalProperties": False},
    },
}
assert records == []
requested = {"id": "call_1", "name": "record_interest", "arguments": '{"email":"learner@example.com"}'}
allowed = {"record_interest": record_interest}  # An explicit registry of permitted functions.
if requested["name"] not in allowed:
    raise ValueError("Unknown tool")
result = allowed[requested["name"]](**json.loads(requested["arguments"]))
tool_message = {"role": "tool", "tool_call_id": requested["id"], "content": json.dumps(result)}
print(tool_message)
assert records == ["learner@example.com"]

**Check:** No. Only the explicit dispatcher call changes records. Keep the assistant tool-call message before the matching tool result in a full conversation.

**Change one thing:** Change the requested name to an unknown function and observe the error. Then add a harmless second tool to the registry.

## 5. An agent loop must make progress and stop

A loop repeats model/tool steps until a final answer or a configured limit. This fixed script of decisions shows the order without executing generated code or calling a model.

**Predict:** Which decision ends the loop?

In [ ]:
decisions = iter([
    {"kind": "tool", "name": "add", "a": 12, "b": 30},
    {"kind": "answer", "text": "The sum is 42."},
])
events = []
for turn in range(3):  # Bound the number of steps.
    decision = next(decisions)
    if decision["kind"] == "tool":
        if decision["name"] != "add":
            raise ValueError("Unknown tool")
        events.append({"tool_result": decision["a"] + decision["b"]})
    else:
        events.append({"final": decision["text"]})
        break
else:
    raise RuntimeError("No final answer within the limit")
print(events)
assert events[0]["tool_result"] == 42

**Check:** The answer decision ends it. A loop that only keeps requesting tools needs a bound and an explicit failure path.

**Change one thing:** Add another tool step before the answer. Keep a finite turn limit and explain how the stopping rule changes.

## 6. Await vs. concurrent work

Calling an async function creates a coroutine. Await executes/waits for it. gather schedules independent coroutines and returns results in input order. Notebook cells support top-level await.

**Predict:** Will results follow completion order or input order?

In [ ]:
import asyncio
async def work(label, delay):
    await asyncio.sleep(delay)  # Yield control while waiting.
    return label
results = await asyncio.gather(work("A", 0.002), work("B", 0))
print(results)
assert results == ["A", "B"]

**Check:** Input order. In a standalone script, wrap work in async def main and call asyncio.run(main()); do not do that inside a running notebook loop.

**Change one thing:** Run the two functions with separate awaits. Explain why a dependent stage cannot start until it has its input.

## 7. Streaming sends partial progress

A generator yields multiple updates. A streamed API can include multiple event types, so a UI should select the text events rather than assume every event is text.

**Predict:** How many text chunks are selected?

In [ ]:
events = [
    {"type": "status", "value": "started"},
    {"type": "text", "value": "Hello "},
    {"type": "text", "value": "learner"},
]
def text_chunks(items):
    for event in items:
        if event["type"] == "text":
            yield event["value"]
chunks = list(text_chunks(events))
print(chunks, "→", "".join(chunks))
assert "".join(chunks) == "Hello learner"

**Check:** Two. yield preserves progress; return ends an ordinary function.

**Change one thing:** Insert a non-text event between the chunks. Confirm it does not become displayed answer text.

## 8. Structured output validates shape

Pydantic models name the fields you expect and check constraints. This helps callers use results reliably, but a valid field can still contain a false claim.

**Predict:** Will confidence=2 pass the constraint?

In [ ]:
from pydantic import BaseModel, Field, ValidationError
class Report(BaseModel):
    summary: str
    confidence: float = Field(ge=0, le=1)
report = Report(summary="Fictional result", confidence=0.8)
print(report.model_dump())
try:
    Report(summary="Bad shape", confidence=2)
except ValidationError:
    print("Invalid confidence rejected")
else:
    raise AssertionError("Constraint was not enforced")

**Check:** No. The value must be between zero and one. Validation checks the constraint, not the truth of the summary.

**Change one thing:** Add a field named follow_up_questions with type list[str]. Construct a valid object using it.

## 9. Evaluate, give feedback, and bound revision

A worker produces a draft and an evaluator judges it against criteria. Feedback becomes input for another attempt. The evaluator here is a small deterministic teaching rule, not a general quality checker.

**Predict:** Is the first draft acceptable? How many attempts will be allowed?

In [ ]:
def evaluate(text):
    return {"accepted": "example:" in text.lower(), "feedback": "Include an example."}
attempts = []
for attempt in range(2):
    draft = "A list stores ordered values." if attempt == 0 else "A list stores ordered values. Example: [1, 2]"
    decision = evaluate(draft)
    attempts.append(decision["accepted"])
    if decision["accepted"]:
        break
print(attempts, draft)
assert attempts == [False, True]

**Check:** The first is rejected and the second accepted. Unlike the Lab 3 one-retry source, this teaching example evaluates the revision too.

**Change one thing:** Make the second draft fail too. Add an explicit message when the finite retry limit is reached.

## 10. Plan → search → synthesize

The research workflow separates a search plan, collected findings, and a report. Use fixed fictional snippets first so you can inspect how evidence reaches the writer. No current facts are being researched here.

**Predict:** Will the report receive the original query as well as the findings?

In [ ]:
from dataclasses import dataclass
@dataclass
class SearchItem:
    reason: str
    query: str
query = "How should a beginner practice tool calls?"
plan = [SearchItem("Understand arguments", "tool argument schema"),
        SearchItem("Understand execution", "tool dispatcher")]
fixture = {"tool argument schema": "A schema describes arguments.",
           "tool dispatcher": "Application code executes an allowed function."}
findings = [fixture[item.query] for item in plan]
report = {"query": query, "summary": " ".join(findings), "follow_up_questions": ["How do we test invalid arguments?"]}
print(report)
assert len(findings) == 2 and "Application" in report["summary"]

**Check:** Yes. Plans choose questions, search collects material, and synthesis uses it. Delivery is a separate side effect.

**Change one thing:** Add a third search plan item and a matching fixture. Explain how the original query keeps synthesis focused.

## 11. Drafting and delivering are separate steps

An agent that formats text is different from a tool that sends email or notifications. This exercise records a delivery intention in memory only.

**Predict:** Will constructing this draft contact anyone?

In [ ]:
outbox = []
draft = {"to": "learner@example.com", "subject": "Practice report", "body": "Fictional findings."}
assert outbox == []
def dry_run_send(message):
    outbox.append({"status": "dry_run", "subject": message["subject"]})
    return {"status": "dry_run"}
result = dry_run_send(draft)
print(result, outbox)
assert len(outbox) == 1

**Check:** No. Neither this draft nor this in-memory dry run sends anything. A provider send/post call is the delivery boundary.

**Change one thing:** Format the body as HTML but keep dry_run_send unchanged. Explain where real delivery would begin.

## 12. A callback receives a message and history

A UI invokes your function and displays its return value. Earlier turns are an input, not automatic model memory. This practice calls the callback directly without starting a server.

**Predict:** What changes when history contains two old messages?

In [ ]:
def chat(message, history):
    return f"You asked: {message}. Earlier messages: {len(history)}."
first = chat("Hello", [])
second = chat("Explain more", [{"role":"user", "content":"Hello"}, {"role":"assistant", "content":first}])
print(first, second, sep="\n")
assert "Earlier messages: 2" in second

**Check:** The count becomes two. Gradio wiring is gr.ChatInterface(chat, type="messages"); launch starts the server.

**Change one thing:** Make the callback return an uppercase version of the question. Compare returning a value with printing it.

## 13. Parameterized SQL and isolated data

SQLite stores records. Parameterized values keep data separate from SQL syntax. We use :memory: so the exercise neither deletes nor changes the source databases.

**Predict:** What happens when you query a city that is not present?

In [ ]:
import sqlite3
with sqlite3.connect(":memory:") as connection:
    connection.execute("CREATE TABLE cities (city TEXT PRIMARY KEY, price REAL)")
    connection.execute("INSERT INTO cities VALUES (?, ?)", ("london", 299))
    def get_city_price(city):
        row = connection.execute("SELECT price FROM cities WHERE city = ?", (city.lower(),)).fetchone()
        return row[0] if row else None
    print(get_city_price("London"), get_city_price("Unknown"))
    assert get_city_price("London") == 299 and get_city_price("Unknown") is None

**Check:** None. A database lookup and a model interpretation of its result are separate stages.

**Change one thing:** Insert another fictional city and query it. Explain why source code deleting tickets.db should not run just to inspect an import.

## 14. A recurring engine is different from a UI

The UI displays state. An engine repeatedly invokes workers. A finite practice loop makes the scheduling visible without leaving a background job running.

**Predict:** How many worker invocations happen with two rounds and four traders?

In [ ]:
import asyncio
calls = []
async def trader_run(name):
    calls.append(name)
    await asyncio.sleep(0)
    return f"{name}: dry run"
for round_number in range(2):
    results = await asyncio.gather(*(trader_run(name) for name in ["A", "B", "C", "D"]))
    print(round_number + 1, results)
assert len(calls) == 8

**Check:** Eight. Real recurring runs can accumulate API usage and change stored account state.

**Change one thing:** Add a market-open boolean that skips a round. Explain why the source while True needs an intentional stop method.

## 15. Configuration fills named placeholders

The crew entry point supplies inputs, and YAML descriptions contain placeholders. Configuration names must match the inputs used at runtime.

**Predict:** What happens if motion is not supplied?

In [ ]:
description = "Propose an argument for: {motion}."
inputs = {"motion": "Beginners should practice offline first"}
rendered = description.format(**inputs)
print(rendered)
assert inputs["motion"] in rendered
try:
    description.format(**{})
except KeyError:
    print("Missing placeholder detected")

**Check:** Formatting raises KeyError here. Crew configuration and project environment are separate from the root project.

**Change one thing:** Add a second placeholder named audience and provide its value. Identify matching input keys in the linked source main.py.

## 16. Request, status, and parsed body

An HTTP client sends a request and returns a status plus response body. Provider requests are network side effects; a fixture lets you practice handling success/failure without sending one.

**Predict:** Will a non-200 response be treated as success here?

In [ ]:
fixtures = [{"status": 200, "body": {"result": "fictional success"}},
            {"status": 401, "body": {"error": "authentication rejected"}}]
for response in fixtures:
    if response["status"] >= 400:
        print("Failure status:", response["status"])
    else:
        print("Parsed result:", response["body"]["result"])
assert fixtures[1]["status"] == 401

**Check:** Status values below 400 enter this fixture success branch; 401 is rejected. Use the provider response to distinguish causes.

**Change one thing:** Add a 429 fixture and distinguish rate limiting from authentication. Explain why real calls need timeouts.

## Rebuild from memory

1. Recreate the main data-flow step in a new scratch cell.
2. Add one case that should fail and identify its exception/stop condition.
3. Compare your version with the linked source cells or functions.
4. Explain where a real model or service would enter, and what it could change.

For deliberate original-source execution, see the shared [configuration guide](../../../../configuration.md). All provider/transport integrations remain UNVERIFIED at runtime.